# Removing Duplicates

In **Notebook 05**, we removed exact duplicates (rows where every column is identical) and discovered that **natural key duplicates** remain — rows that share the same key (`school_urn` or `pupil_id`) but differ in other columns.

These are the harder duplicates to resolve because they require a **business rule** to decide which row to keep. This notebook covers:

1. **Detect** natural key duplicates and understand what differs between them
2. **Choose** a business rule for selecting the "winning" row
3. **Apply** `ROW_NUMBER` with meaningful ordering to deduplicate both schools and pupils

> **Note:** All data in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals.

> **Prerequisite:** Run **Notebook 03 — Combining Snapshots** first to create the silver tables.

In [0]:
-- Setup: recreate the exact-deduped schools view (technique from Notebook 05)
-- This removes carbon-copy rows so we can focus on natural key conflicts
CREATE OR REPLACE TEMP VIEW schools_no_exact_dupes AS
WITH cte_hashed AS (
  SELECT CONCAT_WS('|', *) AS row_hash, *
  FROM catalog_40_copper_analyst_training.silver.schools_combined
)
,cte_numbered AS (
  SELECT ROW_NUMBER() OVER (PARTITION BY row_hash ORDER BY row_hash) AS occurrence, *
  FROM cte_hashed
)
SELECT * EXCEPT (row_hash, occurrence)
FROM cte_numbered
WHERE occurrence = 1;

### Checking foreign key integrity

Joins only work correctly when the foreign key values in one table **actually exist** in the referenced table. When they don’t, you have **orphaned foreign keys** — rows that point to a record that isn’t there.

This can happen for many reasons:

* A school closed or was merged and removed from the reference table, but pupils still reference its URN
* A data load failed partway through, populating one table but not the other
* A manual data entry error introduced a URN that was never valid

Orphaned foreign keys are dangerous because they **silently drop rows** from inner joins. If you join pupils to schools and a pupil references a school that doesn’t exist, that pupil simply vanishes from your results — with no error or warning.

The pattern below uses a `LEFT JOIN` with a `WHERE ... IS NULL` filter to surface any orphaned records. In our synthetic data, pupil P019 (Sophie) references school URN 999999, which doesn’t appear in the schools table.

In [0]:
-- Find orphaned foreign keys: pupils referencing a school that doesn't exist
SELECT
  p.pupil_id
  ,p.first_name
  ,p.last_name
  ,p.school_urn as orphaned_school_urn
FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024 p
LEFT JOIN catalog_40_copper_analyst_training.messy_data.schools_autumn_2024 s
  ON p.school_urn = s.school_urn
WHERE s.school_urn IS NULL;

### Handling orphaned records — a business decision

Once you’ve identified orphaned foreign keys, you need to decide what to do with them. In some cases you may choose to **exclude these rows** from your analysis for data quality purposes — for example, if pupils reference schools that no longer exist in your reference data, including them could distort school-level aggregations or introduce misleading “unknown” categories.

However, removing data is never a purely technical decision. It is a **business decision** that should be:

* **Documented clearly** — record *what* was removed, *why*, and *how many rows* were affected
* **Proportionate** — if orphaned rows represent a significant share of the data, exclusion could introduce bias
* **Reversible** — filter rows out rather than deleting them, so the decision can be revisited
* **Communicated** — stakeholders should know that certain records were excluded and understand the impact

> **Good practice:** Add a comment or markdown cell to your notebook explaining the rationale whenever you filter out data. Future you (or a colleague) will thank you for it.

### Going further: discovering foreign key relationships dynamically

In practice, tables don’t always have formally declared foreign key constraints — especially in data lake environments. You can use `INFORMATION_SCHEMA` to discover columns that **share the same name** across tables, which are strong candidates for foreign key relationships.

The query below finds columns that appear in both the pupils and schools tables, suggesting a join relationship.

In [0]:
-- Find columns that share names between pupils and schools tables
-- These are strong candidates for foreign key / join relationships
SELECT
  p.column_name
  ,p.data_type as pupils_type
  ,s.data_type as schools_type
  ,CASE
    WHEN p.data_type = s.data_type THEN 'Types match - likely join key'
    ELSE 'Type mismatch - investigate'
  END as assessment
FROM (
  SELECT column_name, data_type
  FROM catalog_40_copper_analyst_training.information_schema.columns
  WHERE table_schema = 'messy_data' AND table_name = 'pupils_autumn_2024'
) p
JOIN (
  SELECT column_name, data_type
  FROM catalog_40_copper_analyst_training.information_schema.columns
  WHERE table_schema = 'messy_data' AND table_name = 'schools_autumn_2024'
) s ON p.column_name = s.column_name;

## Summary — What did we learn?

Natural key duplicates are rows that share the same identifying key but differ in other columns. Unlike exact duplicates, resolving them requires a **business decision**:

1. **Detect** — `GROUP BY` the natural key and filter to `HAVING COUNT(*) > 1`
2. **Inspect** — view the actual conflicting rows to understand what differs
3. **Choose** — define a business rule (latest snapshot, most complete, authoritative source)
4. **Apply** — `ROW_NUMBER() OVER (PARTITION BY key ORDER BY rule)` keeps the winning row

### What's next

* **Notebook 07 — Standardising Fields and Labels** will clean the values themselves (casing, whitespace, abbreviations)
* **Notebook 08 — Building the Gold Layer** will apply all steps end-to-end and write to gold